# mojoexporter
Python exporter for Mojo

In [ ]:
#| default_exp mojoexporter

In [ ]:
#| hide 
from nbdev.showdoc import *

In [ ]:
#| export
import subprocess
from pathlib import Path
from nbdev.config import get_config
from nbdev.process import Processor

In [ ]:
#| export
class MojoExport(Processor):
    def end(self):
        cells = [
            cell
            for cell in self.nb.cells
            if cell.cell_type == "code"
            and "export_mojo" in cell.directives_
            and cell.source.lstrip().startswith("%%mojo")
        ]

        if not cells:
            return

        module = next(
            (
                cell.directives_["default_exp"]
                for cell in self.nb.cells
                if "default_exp" in cell.directives_
            ),
            None,
        )

        if not module:
            raise ValueError("Mojo exports require a #| default_exp cell")

        bodies = []

        for cell in cells:
            source = cell.source.lstrip()
            body = source.split("\n", 1)[1].strip()
            bodies.append(body)
            cell.cell_type = "raw"

        destination = Path(get_config().lib_path)
        path = destination / f"{module.replace('.', '/')}.mojo"
        path.parent.mkdir(parents=True, exist_ok=True)

        source = "# AUTOGENERATED! DO NOT EDIT!\n\n"
        source += "\n\n".join(bodies)
        source += "\n"

        path.write_text(source, encoding="utf-8")
        print(f"Exported {path}")

In [ ]:
# TODO Test MojoExport in this nb

In [ ]:
#| export
@register_cell_magic
def mojo(line, cell, wdir='/tmp/temp.mojo'):
    project_root = Path(get_config().config_path)
    Path(wdir).write_text(synth_main(cell))

    subprocess.run(
        ['pixi', 'run', 'mojo', 'run', '-I', str(project_root), wdir],
        cwd=project_root,
        check=True,
    )

In [ ]:
%%mojo
def main():
    print(1+1)

2


In [ ]:
import nbdev; nbdev.nbdev_export()

Exported /app/data/mojotools/mojotools/core.mojo
